# 🐳 Chapter 11: Container Technologies: Docker and Docker Compose

📖 Book: Chapter 11, pp. 430–465 · ⏱️ 2–3 hours · 🧱 Needs: Ch 4, 10 · Level: Intermediate

---

> **🎯 Key idea:** Multi-stage `python:3.12-slim` Dockerfiles, non-root `USER`, `HEALTHCHECK`, and `docker compose` v2 deliver fast, secure, portable containers.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** `docker-compose` (v1) + `version: '3'`  
> **🆕 Now (2026):** `docker compose` (v2, no hyphen, no `version:`)  
> **Why:** Built into Docker CLI; `version:` obsolete

> **📜 Then (book, ~2020):** `python:3.7.3-alpine`  
> **🆕 Now (2026):** `python:3.12-slim` + multi-stage build  
> **Why:** Avoids slow musl wheel compilation

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
Your Flask app runs fine on Python 3.12 locally, but crashes on a teammate's machine running Python 3.9 and a different Postgres version. You need a single reproducible package for the app, its dependencies, and its backing database. Docker and `docker compose` v2 isolate the runtime while keeping secrets out of image layers.

### Concept: Layer Caching & Multi-Stage
**What:** Ordered `Dockerfile` build stages  
**Why:** Keeps final images small and builds fast  
**How:** Compile wheels in builder; copy only `/wheels` + app  
**When:** Every containerized Python service

📖 *Book listing:* Containerized Flask `Dockerfile` (pp. 433–438)

**Plain-language analogy:** A VM is a standalone house with its own plumbing and electricity (guest OS); a container is an apartment sharing the building's utilities (host kernel) while keeping its own locked doors.

**Walkthrough:** The book packages Flask with `WORKDIR`, `COPY requirements.txt`, `RUN pip install`, and `CMD`. Modernizing to `python:3.12-slim`, multi-stage builds, and non-root `USER` shrinks and secures the image.

### Concept: `0.0.0.0` Binding & `sops`
**What:** Container network & secret injection  
**Why:** `127.0.0.1` is unreachable from host `-p`  
**How:** Bind `0.0.0.0`; decrypt `sops` into env at runtime  
**When:** All Dockerized web APIs

📖 *Book listing:* Postgres + Redis Compose Stack & `sops` (pp. 446–463)

**Plain-language analogy:** Docker layer caching is like building a burger from bottom to top: put slow-changing ingredients (`requirements.txt`) on the bottom so changing the top sauce (app code) doesn't re-cook everything.

**Walkthrough:** The book connects `app`, `worker`, `db`, and `redis` on a Compose network with a persistent volume (`dbdata`) and decrypts DB passwords at runtime using `sops -d`.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Generate a modern python:3.12-slim Dockerfile
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Generate a modern python:3.12-slim Dockerfile
def generate_dockerfile(base_image: str = 'python:3.12-slim', app_file: str = 'app.py', port: int = 5000) -> str:
    return '\n'.join([
        f'FROM {base_image}',
        'WORKDIR /app',
        'COPY requirements.txt .',
        'RUN pip install --no-cache-dir -r requirements.txt',
        'COPY . .',
        'RUN useradd --create-home appuser && chown -R appuser /app',
        'USER appuser',
        f'EXPOSE {port}',
        f'CMD ["python", "{app_file}"]',
    ])

df_text = generate_dockerfile()
print(df_text)

FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
RUN useradd --create-home appuser && chown -R appuser /app
USER appuser
EXPOSE 5000
CMD ["python", "app.py"]


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Parse image ref & explain container exit codes
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Parse image ref & explain container exit codes
def parse_image_ref(ref: str) -> tuple[str, str]:
    return tuple(ref.split(':', 1)) if ':' in ref else (ref, 'latest')

def explain_exit_code(code_: int) -> str:
    return {0: 'Success', 1: 'App error', 127: 'Command not found', 137: 'SIGKILL / OOMKilled'}.get(code_, 'Unknown')

print(parse_image_ref('python:3.12-slim'), '|', explain_exit_code(137))

('python', '3.12-slim') | SIGKILL / OOMKilled


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Generate a .dockerignore file
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Generate a .dockerignore file
DOCKERIGNORE = '\n'.join(['.git/', '.venv/', '__pycache__/', '*.pyc', '.env', '*.secrets'])
print(DOCKERIGNORE)

.git/
.venv/
__pycache__/
*.pyc
.env
*.secrets


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: Dockerfile Linter, `yaml.safe_load` Compose Validator & Docker SDK

#### Example 4: Intermediate Example 1: Pure-Python Dockerfile linter
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Pure-Python Dockerfile linter
def lint_dockerfile(content: str) -> list[str]:
    warnings: list[str] = []
    lines = [l.strip() for l in content.splitlines() if l.strip() and not l.strip().startswith('#')]
    if any(':latest' in l or ':' not in l for l in lines if l.startswith('FROM ')):
        warnings.append('Pin explicit base image tag (avoid :latest)')
    if not any(l.startswith('USER ') and 'root' not in l for l in lines):
        warnings.append('Run container as non-root USER')
    if any('SECRET=' in l or 'PASSWORD=' in l for l in lines):
        warnings.append('Never hardcode secrets in Dockerfile')
    return warnings

print('Good Dockerfile:', lint_dockerfile(df_text))
print('Bad Dockerfile:', lint_dockerfile('FROM python:latest\nENV PASSWORD=123\n'))

Good Dockerfile: []
Bad Dockerfile: ['Pin explicit base image tag (avoid :latest)', 'Run container as non-root USER', 'Never hardcode secrets in Dockerfile']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 5: Pure-Python docker compose v2 validator (no external deps needed)
*Expected output:* Printed labelled output below.

In [5]:
# Pure-Python docker compose v2 validator (no external deps needed)
def validate_compose(compose_dict: dict) -> list[str]:
    errors: list[str] = []
    if 'version' in compose_dict:
        errors.append('Remove deprecated top-level version key (Compose v2)')
    services = compose_dict.get('services', {})
    for name, svc in services.items():
        for dep in svc.get('depends_on', []):
            if dep not in services:
                errors.append(f'{name}: depends_on unknown service {dep!r}')
    return errors

print('Pure validator ready:', validate_compose({'services': {'app': {'depends_on': ['db']}, 'db': {}}}))

Pure validator ready: []


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 6: External: Parse compose YAML with PyYAML + Docker SDK check (requires: pip install pyyaml docker)
*Expected output:* Simulated fallback if optional package is missing.

In [6]:
# External: Parse compose YAML with PyYAML + Docker SDK check (requires: pip install pyyaml docker)
COMPOSE_YAML = '''
services:
  app:
    build: .
    depends_on: [db]
  db:
    image: postgres:16-alpine
'''
try:
    import yaml
    parsed_compose = yaml.safe_load(COMPOSE_YAML)
    print('Compose v2 errors:', validate_compose(parsed_compose))
except ImportError:
    print('PyYAML not installed; install via: pip install pyyaml')

Compose v2 errors: []


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Safe `docker compose` / `docker run` Command Builder** avoiding `shell=True`.

#### Example 7: def build_docker_run_cmd(image: str, port_map: str, env_vars: dict[str, str]) -> list[str]:
*Expected output:* Printed labelled output below.

In [7]:
def build_docker_run_cmd(image: str, port_map: str, env_vars: dict[str, str]) -> list[str]:
    cmd = ['docker', 'run', '--rm', '-d', '-p', port_map]
    for k, v in env_vars.items():
        cmd.extend(['-e', f'{k}={v}'])
    cmd.append(image)
    return cmd

print('Safe cmd:', build_docker_run_cmd('my-app:1.0.0', '5000:5000', {'APP_ENV': 'prod'}))

Safe cmd: ['docker', 'run', '--rm', '-d', '-p', '5000:5000', '-e', 'APP_ENV=prod', 'my-app:1.0.0']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 8: Real Tool Demo: docker & docker compose v2 check (guarded with shutil.which)
*Expected output:* Printed labelled output below.

In [8]:
# Real Tool Demo: docker & docker compose v2 check (guarded with shutil.which)
import shutil, subprocess
if shutil.which('docker'):
    print('Docker:', subprocess.run(['docker', '--version'], capture_output=True, text=True, check=True).stdout.strip())
    print('Compose v2:', subprocess.run(['docker', 'compose', 'version'], capture_output=True, text=True, check=True).stdout.strip())
else:
    print('docker not installed; skipped')

Docker: Docker version 29.10.1, build c48360f


Compose v2: Docker Compose version v5.8.0


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Binding `127.0.0.1` in container** -> Connection refused on `localhost:5000`. ✅ Bind Flask/uvicorn to `0.0.0.0`.
- ❌ **Missing `.dockerignore`** -> Copies `.git/` & `.env` into image. ✅ Exclude `.git`, `.venv`, `.env`.
- ❌ **Exit code `137`** -> Container OOMKilled (`SIGKILL`). ✅ Increase memory limit or optimize RAM.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement has_healthcheck and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [9]:
# Starter Exercise Cell (implement has_healthcheck and run to verify!)
def _run_starter_checks():
    assert has_healthcheck('FROM python:3.12-slim\nHEALTHCHECK CMD true') is True
def has_healthcheck(dockerfile: str) -> bool:
    raise NotImplementedError('Implement has_healthcheck')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement has_healthcheck


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** Why prefer `python:3.12-slim` over `-alpine` for Python apps?

<details>
<summary>✅ Answer 1</summary>

`-slim` uses `glibc` so binary wheels install fast without compiling C extensions from source.

</details>

**Q2:** Why should `COPY requirements.txt .` precede `COPY . .`?

<details>
<summary>✅ Answer 2</summary>

Preserves Docker layer cache so changing app code doesn't re-install dependencies.

</details>

**Q3:** Why must containerized web apps bind to `0.0.0.0`?

<details>
<summary>✅ Answer 3</summary>

`127.0.0.1` is loopback inside the container only; `0.0.0.0` accepts host-forwarded traffic.

</details>

**Q4:** What changed from `docker-compose` v1 to `docker compose` v2?

<details>
<summary>✅ Answer 4</summary>

No hyphen (`docker compose`), built into Docker CLI, and the top-level `version:` key is obsolete.

</details>

**Q5:** How does `sops` protect secrets?

<details>
<summary>✅ Answer 5</summary>

Encrypts secret values at rest with PGP/KMS and decrypts only in memory at runtime.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 4**: Match lines starting with `'FROM '` and split on whitespace (`line.split()[1]`).

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [10]:
# Complete Solutions + Assert Verification
def has_healthcheck(dockerfile: str) -> bool:
    return any(l.strip().startswith('HEALTHCHECK ') for l in dockerfile.splitlines())

def parse_dockerfile_stages(content: str) -> list[str]:
    return [l.split()[1] for l in content.splitlines() if l.strip().startswith('FROM ')]

def audit_container_repo(dockerfile_txt: str, compose_dict: dict) -> dict:
    df_w = lint_dockerfile(dockerfile_txt)
    comp_e = validate_compose(compose_dict)
    return {'ready': not df_w and not comp_e, 'dockerfile_warnings': df_w, 'compose_errors': comp_e}

assert has_healthcheck('FROM python:3.12-slim\nHEALTHCHECK CMD true') is True
assert parse_dockerfile_stages('FROM python:3.12-slim AS builder\nFROM python:3.12-slim') == ['python:3.12-slim', 'python:3.12-slim']
assert audit_container_repo(df_text, {'services': {'db': {}}})['ready'] is True
print('All Chapter 11 assertions PASSED!')

All Chapter 11 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `docker build -t app:v1 .` — Build container image (e.g. `docker build --no-cache -t app:1.0 .`)
- `docker compose up -d` — Start v2 multi-service stack (e.g. `docker compose logs -f`)
- `sops -d env.secrets` — Decrypt runtime secrets (e.g. `source <(sops -d env.secrets)`)

### ✅ What You Can Now Do
- Author and lint multi-stage `python:3.12-slim` Dockerfiles with non-root `USER` and `HEALTHCHECK`.
- Validate `compose.yaml` (v2) definitions with `yaml.safe_load()` and understand `0.0.0.0` binding.
- Verify local container tooling safely with `docker` and `docker compose` CLI checks.

### ➡️ Next Chapter
Chapter 12 builds on this by scheduling, scaling, and self-healing those containers across multi-node Kubernetes clusters.